# 📘 부록: 이미지 생성 모델에 LoRA로 애니메이션 캐릭터 학습시키기

교안 01에서 언어 모델의 선형 층에 붙인 LoRA 어댑터는 이미지 생성 모델의 선형 층에도 같은 `LoraConfig`로 붙습니다. 이 부록은 이미지 생성 모델 `black-forest-labs/FLUX.2-klein-base-4B`이 처음 보는 오리지널 캐릭터 초롱이를 그림 42장으로 학습시켜, 학습 그림에 없는 장면(우주에서 헤엄치기, 옥상에서 수박 먹기, 바닷속, 열기구)과 다른 화풍(수채화, 연필 스케치)에서도 그 캐릭터를 그리게 만듭니다. 캐릭터를 배우면서 평범한 작은 생물·개·사람 그림 능력(일반화 능력)을 잃지 않도록, 저장한 체크포인트 가운데 알맞은 것을 고르는 것까지 합니다.

**실행 환경:** 교안과 같은 RunPod 파드(GPU 메모리 24GB 이상)에서 실행합니다. 모델 파일은 약 16GB이고, 처음 불러올 때 내려받습니다.

## 1. Diffusion 모델과 DreamBooth LoRA

**Diffusion 모델(diffusion model)**
- 프롬프트(문장)를 받아 그림을 만드는 이미지 생성 모델의 한 종류입니다. 이 부록의 FLUX.2 [klein] base 4B(Black Forest Labs, Apache 2.0)도 Diffusion 모델입니다.
- **순방향(forward process):** 깨끗한 그림에 잡음을 섞습니다. 섞는 비율이 100%가 되면 잡음만 남습니다. 학습할 때는 비율을 무작위로 골라 한 번에 섞어 학습 문제를 만듭니다(아래 플로 매칭).
- **역방향(reverse process):** 잡음에서 시작해 잡음을 조금씩 걷어 내며 그림을 만듭니다. 매번 걷어 낼 방향은 잡음 제거 신경망이 예측합니다. 그림을 만들 때 쓰는 방향입니다.
- 잡음 제거 신경망으로 Stable Diffusion 1.5·SDXL은 UNet을, FLUX 계열은 트랜스포머를 씁니다.

<img src="images/27_diffusion_forward.png" width="1050" alt="순방향: 깨끗한 초롱이 그림에 잡음을 30%, 70%, 100% 섞어 가면 잡음만 남습니다. 학습할 때 쓰는 방향입니다.">

<img src="images/28_diffusion_reverse.png" width="1050" alt="역방향: 잡음에서 시작해 잡음을 조금씩 걷어 내 10번, 20번 뒤 초롱이가 드러나고 30번 뒤 선명한 그림이 됩니다. 그림을 만들 때 쓰는 방향입니다.">


**FLUX.2 [klein]의 세 부분**

| 부분 | 하는 일 | 학습 |
|---|---|---|
| 텍스트 인코더(text encoder) | 프롬프트를 토큰마다 벡터로 바꿉니다(언어 모델 Qwen3-4B) | 하지 않음 |
| 트랜스포머(transformer) | 잡음 제거 신경망(파라미터 약 40억 개)입니다. 잡음 섞인 그림과 프롬프트 벡터를 받아 걷어 낼 방향을 예측합니다 | LoRA 어댑터만 |
| VAE(Variational Autoencoder) | 인코더가 그림을 작은 잠재 표현(latent)으로 줄이고, 디코더가 다시 그림으로 되돌립니다 | 하지 않음 |

- 트랜스포머는 그림 대신 VAE가 줄인 잠재 표현으로 계산해 계산량이 적습니다.
- 그림 하나를 만들 때 트랜스포머를 30번 거쳐 잡음을 걷어 낸 뒤, VAE 디코더로 그림을 만듭니다.


<img src="images/30_diffusion_lora.png" width="1050" alt="Diffusion 모델의 세 부분(텍스트 인코더, 트랜스포머, VAE 디코더)과 LoRA 어댑터를 붙이는 자리. 프롬프트 벡터와 잡음이 트랜스포머를 30번 거친 뒤 VAE 디코더를 지나 초롱이 그림이 됩니다.">


**플로 매칭(flow matching) 학습**
- 트랜스포머를 학습시키는 방법입니다. 학습 그림의 잠재 표현에 잡음을 섞고, 그 섞인 표현에서 '잡음 쪽으로 가는 방향'을 맞히게 합니다. Diffusion 모델을 학습시키는 방법의 하나로, 그림과 잡음을 직선으로 섞습니다.
- 학습 그림의 잠재 표현을 $x_0$, 같은 모양의 표준정규 잡음을 $\varepsilon$, 잡음 비율을 $\sigma$(0~1에서 균등하게 뽑음)라 하면 섞인 표현은 다음과 같습니다.

$\displaystyle x_\sigma = (1 - \sigma)\, x_0 + \sigma\, \varepsilon$

- $\sigma = 0.25$면 그림 75%와 잡음 25%, $\sigma = 1$이면 잡음만 남습니다. 트랜스포머의 예측 $v_\theta$가 맞혀야 할 방향은 $\varepsilon - x_0$이고, 손실은 그 차이의 제곱 평균(MSE)입니다.

$\displaystyle L = \text{평균}\Bigl(\bigl(v_\theta(x_\sigma, \sigma, \text{프롬프트}) - (\varepsilon - x_0)\bigr)^2\Bigr)$

- 그림을 만들 때는 이 방향을 거꾸로 따라가 $\sigma = 1$(잡음)에서 $\sigma = 0$(그림)으로 갑니다.

**DreamBooth**
- 대상 하나(사람, 물건, 캐릭터)의 그림 몇 장과, 그 대상을 가리키는 낱말이 든 프롬프트로 이미지 생성 모델을 파인튜닝해 그 대상을 새 장면에 그리게 하는 방법입니다([Ruiz et al., 2022, 프로젝트 페이지](https://dreambooth.github.io/)). 원 논문은 모델 전체를 학습했고, 이 부록은 diffusers 공식 예제처럼 LoRA 어댑터만 학습합니다.
- **트리거 단어(trigger word):** 학습한 대상을 불러내는 낱말입니다. 모델이 이미 뜻을 아는 낱말(예: 유명 캐릭터 이름)이면 원래 뜻과 섞이므로, 지어낸 이름을 씁니다. 이 부록은 `Chorong`이고, 부류 낱말 `creature`(작은 생물)와 함께 `Chorong the creature`로 씁니다. 학습 뒤에는 이 말이 초롱이를 가리킵니다.
- **캡션(caption):** 학습 그림마다 붙이는 설명 문장입니다. 그림마다 바뀌는 것(구도, 표정, 배경, 조명)만 적고, 늘 같은 특징(크림색 둥근 몸, 접부채 모양 귀, 초롱이 달린 꼬리, 왼쪽 볼의 점 세 개)은 적지 않습니다. 적지 않은 특징이 `Chorong`에 묶이고, 적은 배경과 구도는 `Chorong`에 묶이지 않아 다른 장면에도 캐릭터를 그릴 수 있습니다.

## 2. 실습 준비

### 2-1. 라이브러리와 경로

교안과 같은 준비 셀에 이미지 생성 라이브러리 `diffusers`를 더합니다. `diffusers`는 Hugging Face의 이미지 생성 모델 라이브러리이고, LoRA는 교안과 같은 `peft`로 붙입니다.

In [ ]:
import json
import math
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

material_dir = Path(".")  # data가 있는 단원 폴더
data_dir = material_dir / "data" / "부록_이미지"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: 이 부록은 GPU(cuda)가 있어야 합니다. CPU로는 학습 한 스텝에 몇 분이 걸립니다
device = torch.device("cuda")
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print("계산 장치:", device, "|", torch.cuda.get_device_name(0), "| GPU 메모리(GB):", round(total_gb, 1))

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다
pd.set_option("display.max_colwidth", None)  # 캡션이 잘리지 않게 합니다

### 2-2. 학습 그림과 캡션

초롱이는 이 과정을 위해 ChatGPT 이미지로 만든 오리지널 캐릭터입니다. 접부채 모양 귀, 청사초롱이 달린 꼬리, 왼쪽 볼에만 있는 점 세 개처럼 학습 전 모델이 설명만으로는 그리지 못하는 특징을 넣었고, 설정화 한 장을 참고 그림으로 붙여 같은 캐릭터를 여러 각도, 표정, 자세, 배경으로 42장(1024 × 1024) 그렸습니다. 캡션은 그림마다 하나씩 `captions.json`에 있습니다.

<img src="data/부록_이미지/chorong_reference.png" width="1050" alt="초롱이 설정화. 같은 캐릭터를 정면, 3/4 측면, 옆면, 뒷모습으로 나란히 그렸습니다. 몸은 크림색의 둥근 작은 생물이고, 귀는 금색 살대가 있는 짙은 청록색 접부채 모양입니다. 꼬리 끝에는 금색 술이 달린 빨강·파랑 비단 초롱이 빛나고, 왼쪽 볼에만 청록색 점 세 개가 있습니다.">

In [ ]:
# 그림 목록, 출처 정보(작품, 라이선스, 저작자 표시), 그림마다 붙인 캡션을 읽습니다
image_dir = data_dir / "chorong"
image_paths = sorted(image_dir.glob("*.jpg"))
source_info = json.loads((data_dir / "sources.json").read_text(encoding="utf-8"))
captions = json.loads((data_dir / "captions.json").read_text(encoding="utf-8"))  # 파일 이름 -> 캡션

print("캐릭터:", source_info["title"], "|", source_info["attribution"])
print("학습 그림 수:", len(image_paths), "| 캡션 수:", len(captions), "| 크기:", Image.open(image_paths[0]).size)

In [ ]:
def show_images(images, titles, n_cols, title):
    # 그림 여러 장을 n_cols칸씩 격자로 그립니다. titles는 그림마다 붙일 짧은 이름입니다
    n_rows = math.ceil(len(images) / n_cols)
    plt.figure(figsize=(n_cols * 2.8, n_rows * 2.9 + 0.5))
    for i in range(len(images)):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(images[i])
        plt.title(titles[i], fontsize=11)
        plt.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def compare_rows(rows, prompt_names, title):
    # rows: [(줄 이름, {프롬프트 이름: 그림}), ...]. 한 줄이 한 모델, 한 열이 한 프롬프트입니다
    grid_images = []
    grid_titles = []
    for row_name, images in rows:
        for name in prompt_names:
            grid_images.append(images[name])
            grid_titles.append(f"{row_name}: {name}")
    show_images(grid_images, grid_titles, len(prompt_names), title)


# 42장 가운데 7장 간격으로 6장과 그 캡션을 봅니다
preview_paths = image_paths[::7]
show_images([Image.open(path) for path in preview_paths], [path.stem for path in preview_paths], 6, "학습 그림 42장 가운데 6장")
caption_rows = []
for path in preview_paths:
    caption_rows.append({"그림": path.stem, "캡션": captions[path.name]})
display(pd.DataFrame(caption_rows))

### 2-3. 모델 불러오기

`Flux2KleinPipeline`은 텍스트 인코더, 트랜스포머, VAE와 생성 순서(스케줄러)를 한 객체로 묶은 파이프라인(pipeline)입니다. 세 모델을 bfloat16(bf16)으로 불러옵니다.

In [ ]:
from diffusers import Flux2KleinPipeline

# 처음 실행하면 허브에서 약 16GB를 내려받습니다(RunPod에서 몇 분)
model_name = "black-forest-labs/FLUX.2-klein-base-4B"
pipe = Flux2KleinPipeline.from_pretrained(model_name, torch_dtype=torch.bfloat16)
pipe.to(device)
pipe.set_progress_bar_config(disable=True)  # 그림마다 찍히는 진행 막대를 끕니다

# 세 부분의 파라미터 수(pipe.components: 부분 이름 -> 모델)
part_rows = []
for part_name in ["text_encoder", "transformer", "vae"]:
    part = pipe.components[part_name]
    part_rows.append({"부분": part_name, "클래스": type(part).__name__, "파라미터 수": sum(p.numel() for p in part.parameters())})
part_table = pd.DataFrame(part_rows)
display(part_table)
print("GPU에 올라간 메모리(GB):", round(torch.cuda.memory_allocated() / 1024 ** 3, 1))

### 2-4. 프롬프트를 벡터로 바꾸고 텍스트 인코더 빼기

이 부록에서 쓰는 프롬프트(캡션 42개와 확인 프롬프트 9개)를 처음에 한 번씩만 벡터로 바꿔 두고, 텍스트 인코더는 파이프라인에서 뺍니다. 텍스트 인코더가 차지하던 GPU 메모리(약 8GB)를 학습에 씁니다. 새 프롬프트로 그릴 때는 8절에서 텍스트 인코더를 다시 얹습니다.

- **캐릭터 확인 프롬프트:** 학습 그림(실내, 숲·정원, 해변, 눈밭, 밤거리의 3D 애니메이션)에 없는 장면 넷(우주에서 헤엄치기, 옥상에서 수박 먹기, 바닷속, 열기구)과, 캐릭터는 그대로 두고 화풍만 바꾼 둘(수채화, 연필 스케치)입니다. 모두 `Chorong the creature`를 넣습니다. 학습 뒤에 캐릭터가 이 장면과 화풍에 맞게 나와야 합니다.
- **일반화 확인 프롬프트:** 트리거 단어 없이 부류만 쓴 `a 3D animated cartoon of a small creature`, 다른 동물 `a 3D animated cartoon of a dog`, 학습과 관계없는 사진 `a photo of a person reading a book in a cafe`입니다. 이 셋은 학습 뒤에도 학습 전과 비슷하게 그려야 합니다.
- **빈 프롬프트:** 그림을 만들 때 프롬프트를 따르는 정도를 높이는 classifier-free guidance(CFG)가 빈 프롬프트의 예측과 비교하는 데 씁니다(`guidance_scale`이 비교 배율).

In [ ]:
character_prompts = {
    "우주 유영": "a 3D animated cartoon of Chorong the creature swimming through outer space among stars and planets",
    "수박 먹기": "a 3D animated cartoon of Chorong the creature eating a big slice of watermelon on a rooftop in summer",
    "바닷속": "a 3D animated cartoon of Chorong the creature swimming underwater in a coral reef with colorful fish",
    "열기구": "a 3D animated cartoon of Chorong the creature riding in a hot air balloon high above the clouds",
    "수채화": "a watercolor painting of Chorong the creature",
    "연필 스케치": "a pencil sketch of Chorong the creature",
}
general_prompts = {
    "부류(작은 생물)": "a 3D animated cartoon of a small creature",
    "다른 동물(개)": "a 3D animated cartoon of a dog",
    "사진(사람)": "a photo of a person reading a book in a cafe",
}
# 두 묶음을 합친 확인 프롬프트 9개
test_prompts = dict(character_prompts)
test_prompts.update(general_prompts)

# 프롬프트마다 (벡터, 토큰 위치 번호)를 한 번 계산해 둡니다
prompt_bank = {}
with torch.no_grad():
    for text in [""] + list(test_prompts.values()) + list(captions.values()):
        prompt_bank[text] = pipe.encode_prompt(text, device=device)

embeds, text_ids = prompt_bank[test_prompts["우주 유영"]]
print("프롬프트 수:", len(prompt_bank), "| 프롬프트 벡터:", tuple(embeds.shape), embeds.dtype, "| 토큰 위치 번호:", tuple(text_ids.shape))

# 텍스트 인코더를 파이프라인에서 빼고 GPU 메모리를 비웁니다
import gc

pipe.text_encoder = None
gc.collect()
torch.cuda.empty_cache()
print("GPU에 올라간 메모리(GB):", round(torch.cuda.memory_allocated() / 1024 ** 3, 1))

- **프롬프트 벡터 (1, 512, 7680):** 프롬프트 하나를 512토큰 길이로 채우고(짧으면 빈 토큰으로), 토큰마다 텍스트 인코더의 세 층(9, 18, 27번째) 출력 2560개씩을 이어 붙인 7680개 숫자입니다.

### 2-5. 그림 생성 함수

같은 프롬프트라도 시작 잡음이 다르면 다른 그림이 나옵니다. 학습 전과 학습 후를 같은 시드(seed)로 그려야 학습이 바꾼 것만 비교할 수 있습니다.

In [ ]:
resolution = 512  # 학습과 생성 모두 512 x 512


def generate(prompt, seed):
    # 미리 계산한 프롬프트 벡터로 그림 한 장을 만듭니다. 같은 seed면 같은 시작 잡음입니다
    prompt_embeds, _ = prompt_bank[prompt]
    negative_embeds, _ = prompt_bank[""]
    generator = torch.Generator(device).manual_seed(seed)
    result = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_embeds,
        height=resolution,
        width=resolution,
        num_inference_steps=30,  # 잡음을 걷어 내는 횟수
        guidance_scale=4.0,  # CFG 배율(이 모델의 기본값)
        generator=generator,
    )
    return result.images[0]

## 3. 학습 전 그림

확인 프롬프트 아홉을 시드 0으로 그립니다. 학습 뒤에 같은 시드로 다시 그려 비교합니다.

In [ ]:
start_time = time.time()
before_images = {}
for name, prompt in test_prompts.items():
    before_images[name] = generate(prompt, seed=0)
print(f"그림 {len(before_images)}장: {time.time() - start_time:.0f}초")

compare_rows([("학습 전", before_images)], list(character_prompts), "캐릭터 확인 프롬프트, 학습 전, 시드 0")
compare_rows([("학습 전", before_images)], list(general_prompts), "일반화 확인 프롬프트, 학습 전, 시드 0")

**결과 읽기**
- **캐릭터 확인 여섯 장:** 학습 전 모델은 `Chorong`을 모르므로 트리거 단어를 뺀 나머지 낱말대로 그립니다. 우주와 바닷속에는 파란 물고기 모양 생물, 수박 장면에는 초록 개구리 같은 생물, 수채화와 연필 스케치에는 용 같은 괴물이 나오고, 접부채 귀나 초롱 꼬리는 하나도 없습니다.
- **일반화 확인 세 장:** 초록 토끼 같은 작은 생물, 주황색 개, 카페에서 책을 읽는 사람 사진입니다. 학습 뒤에도 이 세 장과 비슷해야 합니다.

## 4. LoRA 어댑터 붙이기

### 4-1. 대상 층: target_modules

FLUX.2 [klein]의 트랜스포머는 두 종류의 블록을 쌓았습니다. 교안 01에서 언어 모델의 어텐션 선형 층(`q_proj`, `k_proj`, `v_proj`, `o_proj`)에 어댑터를 붙였듯이, 여기서도 어텐션 계산에 쓰이는 선형 층에 붙입니다(diffusers 공식 학습 예제의 기본 대상과 같습니다).

- **이중 스트림 블록(`transformer_blocks`):** 그림 토큰과 프롬프트 토큰을 따로 변환한 뒤 함께 어텐션합니다. 그림 쪽 선형 층 `to_q`, `to_k`, `to_v`, `to_out.0`에 붙입니다.
- **단일 스트림 블록(`single_transformer_blocks`):** 두 토큰을 한 줄로 이어 처리합니다. Q, K, V와 MLP 입력을 한 번에 만드는 `to_qkv_mlp_proj`와 출력 층 `to_out`에 붙입니다.

In [ ]:
transformer = pipe.transformer
print("이중 스트림 블록:", transformer.config.num_layers, "개 | 단일 스트림 블록:", transformer.config.num_single_layers, "개")

# 블록 하나씩 골라, 그 안의 선형 층 이름과 (출력 크기, 입력 크기)를 봅니다
for block_name in ["transformer_blocks.0", "single_transformer_blocks.0"]:
    print(f"\n[{block_name}]")
    for name, module in transformer.get_submodule(block_name).named_modules():
        if isinstance(module, torch.nn.Linear):
            print(f"  {name}: {tuple(module.weight.shape)}")

### 4-2. LoraConfig와 add_adapter

교안 01은 `get_peft_model`로 모델을 감쌌습니다. diffusers 모델은 `add_adapter(LoraConfig)`로 같은 어댑터를 모델 안에 바로 넣습니다. `LoraConfig`의 뜻은 교안 01과 같습니다.

- **`r` 16, `lora_alpha` 16:** 배율 $\alpha / r = 1$입니다. diffusers 공식 예제의 기본값은 r 4(lora_alpha 4)이고, 이 부록은 각도, 표정, 자세가 다양한 42장을 담으려고 16으로 둡니다.
- **`init_lora_weights="gaussian"`:** $A$를 정규분포 무작위 값, $B$를 0으로 시작합니다. 붙인 직후 $BA = 0$이라 그림은 학습 전과 같습니다.

In [ ]:
from peft import LoraConfig

# 단일 스트림 블록의 출력 층은 이름이 to_out이라, 블록 번호를 붙인 전체 이름으로 적습니다
single_out_layers = [f"single_transformer_blocks.{i}.attn.to_out" for i in range(transformer.config.num_single_layers)]
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0", "to_qkv_mlp_proj"] + single_out_layers,
)
transformer.add_adapter(lora_config)  # 어댑터 이름은 "default"


def trainable_params(model):
    # requires_grad인 값(학습할 어댑터)만 모읍니다. 5절의 옵티마이저가 이 목록을 받습니다
    params = []
    for p in model.parameters():
        if p.requires_grad:
            params.append(p)
    return params


# 학습할 값(어댑터)과 전체 트랜스포머 파라미터 수, 어댑터를 붙인 층 수(층마다 lora_A가 하나)
n_lora = sum(p.numel() for p in trainable_params(transformer))
n_total = sum(p.numel() for p in transformer.parameters())
n_lora_layers = 0
for name, module in transformer.named_modules():
    if name.endswith(".lora_A"):
        n_lora_layers += 1
print(f"어댑터를 붙인 선형 층: {n_lora_layers}개")
print(f"학습 파라미터: {n_lora:,}개 / 전체 {n_total:,}개 ({n_lora / n_total:.2%})")

## 5. 어댑터 학습

### 5-1. 그림을 잠재 표현으로

학습 그림 42장은 학습 내내 바뀌지 않으므로, VAE로 잠재 표현을 한 번만 만들어 둡니다(공식 예제의 `--cache_latents`).

1. 그림을 512 × 512로 줄이고 픽셀 0~255를 -1~1로 바꿉니다.
2. VAE 인코더가 64 × 64 위치마다 숫자 32개인 잠재 표현을 만듭니다(가로세로 1/8).
3. 이웃한 2 × 2 위치를 한 위치로 묶어(patchify) 32 × 32 위치마다 숫자 128개로 바꿉니다. 트랜스포머는 이 1,024개 위치를 그림 토큰으로 봅니다.
4. 숫자 128개마다 VAE에 저장된 평균과 표준편차로 표준화합니다.

In [ ]:
vae = pipe.vae
# 4단계 표준화에 쓰는 평균과 표준편차(VAE의 BatchNorm 층에 저장된 값)
latents_mean = vae.bn.running_mean.view(1, -1, 1, 1).to(device)
latents_std = torch.sqrt(vae.bn.running_var.view(1, -1, 1, 1) + vae.config.batch_norm_eps).to(device)


def images_to_latents(images):
    # PIL 그림 목록 -> (그림 수, 128, 32, 32) 잠재 표현
    latents_list = []
    with torch.no_grad():
        for image in images:
            pixels = np.array(image.convert("RGB").resize((resolution, resolution), Image.LANCZOS))  # (512, 512, 3), 0~255
            pixels = torch.from_numpy(pixels).permute(2, 0, 1).float() / 127.5 - 1  # (3, 512, 512), -1~1
            latents = vae.encode(pixels.unsqueeze(0).to(device, torch.bfloat16)).latent_dist.mode()  # (1, 32, 64, 64)
            latents = Flux2KleinPipeline._patchify_latents(latents)  # (1, 128, 32, 32)
            latents_list.append((latents - latents_mean) / latents_std)
    return torch.cat(latents_list).to(torch.bfloat16)


train_latents = images_to_latents([Image.open(path) for path in image_paths])
print("학습 잠재 표현:", tuple(train_latents.shape), train_latents.dtype)

잡음 비율 $\sigma$에 따라 섞인 표현이 어떻게 보이는지, 첫 학습 그림의 잠재 표현에 잡음을 섞어 VAE로 되돌려 봅니다.

In [ ]:
def latents_to_image(latents):
    # 표준화를 되돌리고 2 x 2 묶음을 풀어 VAE로 그림을 만듭니다(파이프라인의 마지막 단계와 같습니다)
    with torch.no_grad():
        latents = latents * latents_std + latents_mean
        latents = Flux2KleinPipeline._unpatchify_latents(latents.to(torch.bfloat16))
        pixels = vae.decode(latents, return_dict=False)[0]
    return pipe.image_processor.postprocess(pixels, output_type="pil")[0]


torch.manual_seed(0)
x0 = train_latents[:1].float()
noise = torch.randn_like(x0)
sigma_values = [0.0, 0.25, 0.5, 0.75, 1.0]
noisy_images = []
for sigma in sigma_values:
    x_sigma = (1 - sigma) * x0 + sigma * noise  # x_sigma = (1 - sigma) * x0 + sigma * 잡음
    noisy_images.append(latents_to_image(x_sigma))
show_images(noisy_images, [f"sigma {sigma}" for sigma in sigma_values], 5, "잡음 비율에 따라 섞인 표현을 그림으로 되돌린 모습")

- $\sigma = 0.5$까지는 초롱이의 모양이 보이고, $0.75$에서는 귀와 몸의 윤곽만 남고, $1$이면 잡음뿐입니다. 학습은 이 사이 어느 비율에서든 원래 그림 쪽 방향을 맞히게 합니다.

### 5-2. 손실 함수

1절의 식을 그대로 코드로 옮깁니다. 트랜스포머는 잠재 표현을 (그림 수, 1,024, 128)의 토큰 줄로 받으므로 넣기 전에 펼치고(`_pack_latents`), 나온 예측을 다시 (그림 수, 128, 32, 32)로 되돌립니다. 펼치기와 되돌리기, 위치 번호 만들기는 파이프라인의 함수를 그대로 씁니다(공식 학습 예제와 같은 방법).

In [ ]:
def flow_matching_loss(latents, prompt_embeds, text_ids):
    # 플로 매칭 손실(MSE)을 계산합니다
    noise = torch.randn_like(latents)
    sigmas = torch.rand(len(latents), device=device)  # 그림마다 잡음 비율 0~1
    s = sigmas.view(-1, 1, 1, 1).to(latents.dtype)
    noisy = (1 - s) * latents + s * noise  # x_sigma = (1 - sigma) * x0 + sigma * 잡음
    target = noise - latents  # 맞혀야 할 방향 = 잡음 - x0

    latent_ids = Flux2KleinPipeline._prepare_latent_ids(latents).to(device)  # 그림 토큰 1,024개의 (행, 열) 위치 번호
    prediction = transformer(
        hidden_states=Flux2KleinPipeline._pack_latents(noisy),  # (그림 수, 1024, 128)
        timestep=sigmas,
        guidance=None,
        encoder_hidden_states=prompt_embeds,
        txt_ids=text_ids,
        img_ids=latent_ids,
        return_dict=False,
    )[0]
    prediction = Flux2KleinPipeline._unpack_latents_with_ids(prediction, latent_ids)  # (그림 수, 128, 32, 32)

    # MSE = 평균((예측 - 목표) ** 2)
    return ((prediction.float() - target.float()) ** 2).mean()


# 학습 전 손실 한 번(첫 그림과 그 캡션)
first_embeds, first_text_ids = prompt_bank[captions[image_paths[0].name]]
torch.manual_seed(0)
with torch.no_grad():
    print("학습 전 손실(첫 그림, 무작위 sigma 한 번):", round(flow_matching_loss(train_latents[:1], first_embeds, first_text_ids).item(), 4))

### 5-3. 학습과 체크포인트 저장

- **스텝마다 그림 한 장:** 42장 가운데 한 장을 무작위로 골라 그 그림의 캡션과 함께 넣습니다. 잡음 비율도 무작위로 뽑으므로 같은 그림도 스텝마다 다른 문제가 됩니다. 1000스텝은 그림 한 장을 평균 약 23번 보는 양입니다.
- **학습률 1e-4:** diffusers 공식 예제와 HF 블로그의 FLUX.2 [klein] LoRA 값입니다.
- **메모리 줄이기:** 그래디언트 체크포인팅(gradient checkpointing, 순전파의 중간값을 버렸다가 역전파 때 다시 계산)을 켜고, 학습하는 어댑터 값만 float32로 둡니다(작은 갱신이 bf16 반올림에 묻히지 않게).
- **체크포인트:** 지난 시간 에폭마다 체크포인트를 저장했듯이, 250, 500, 750, 1000스텝에서 어댑터를 저장하고 '우주 유영' 프롬프트를 시드 0으로 그려 둡니다. 6절에서 이 가운데 쓸 체크포인트를 고릅니다.

In [ ]:
from diffusers.training_utils import cast_training_params
from peft import get_peft_model_state_dict

lr = 1e-4
n_train_steps = 1000
checkpoint_steps = [250, 500, 750, 1000]
lora_root = output_dir / "chorong_lora"  # 체크포인트마다 step_0250 같은 하위 폴더

torch.manual_seed(0)
transformer.enable_gradient_checkpointing()
cast_training_params(transformer, dtype=torch.float32)  # requires_grad인 값(어댑터)만 float32로
optimizer = torch.optim.AdamW(trainable_params(transformer), lr=lr)

In [ ]:
history_rows = []
snapshots = {0: before_images["우주 유영"]}  # 0: 학습 전
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

transformer.train()
for step in range(1, n_train_steps + 1):
    # 42장 가운데 한 장과 그 그림의 캡션
    index = torch.randint(len(train_latents), (1,)).item()
    caption_embeds, caption_text_ids = prompt_bank[captions[image_paths[index].name]]
    loss = flow_matching_loss(train_latents[index:index + 1], caption_embeds, caption_text_ids)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    history_rows.append({"스텝": step, "손실": loss.item()})

    if step in checkpoint_steps:
        # 어댑터 값만 diffusers 형식(pytorch_lora_weights.safetensors 한 파일)으로 저장합니다
        Flux2KleinPipeline.save_lora_weights(
            save_directory=lora_root / f"step_{step:04d}",
            transformer_lora_layers=get_peft_model_state_dict(transformer),
            transformer_lora_adapter_metadata=lora_config.to_dict(),  # r, lora_alpha, 대상 층(불러올 때 같은 설정으로 붙이려고)
        )
        transformer.eval()
        snapshots[step] = generate(test_prompts["우주 유영"], seed=0)
        transformer.train()
        recent = np.mean([row["손실"] for row in history_rows[-50:]])
        print(f"스텝 {step} | 최근 50스텝 평균 손실 {recent:.4f} | {(time.time() - start_time) / 60:.1f}분")

print(f"학습 시간(중간 그림 포함): {(time.time() - start_time) / 60:.1f}분 | GPU 최대 메모리(GB): {torch.cuda.max_memory_allocated() / 1024 ** 3:.1f}")
for path in sorted(lora_root.glob("step_*/*.safetensors")):
    print(f"{path.parent.name}/{path.name}: {path.stat().st_size / 1024 ** 2:.1f}MB")

**결과 읽기**
- **시간과 메모리:** RunPod RTX PRO 4000(24GB)에서 1000스텝이 중간 그림까지 약 11분, GPU 최대 메모리는 약 8.6GB입니다. 텍스트 인코더를 뺐기 때문에 24GB의 절반도 쓰지 않습니다.
- **어댑터 파일:** 체크포인트마다 약 64MB입니다. 베이스 모델(약 16GB)의 0.4% 정도라 체크포인트를 여러 개 저장해도 부담이 적습니다.

### 5-4. 학습 곡선

스텝마다 그림과 잡음 비율이 바뀌므로 손실 한 점 한 점은 크게 흔들립니다($\sigma$가 1에 가까우면 쉬운 문제, 0에 가까우면 어려운 문제). 50스텝 이동 평균을 함께 그려 흐름을 봅니다.

In [ ]:
history_table = pd.DataFrame(history_rows)
history_table["50스텝 이동 평균"] = history_table["손실"].rolling(50).mean()

plt.figure(figsize=(10, 4))
plt.plot(history_table["스텝"], history_table["손실"], color="lightgray", linewidth=0.8, label="스텝 손실")
plt.plot(history_table["스텝"], history_table["50스텝 이동 평균"], color="tab:red", label="50스텝 이동 평균")
plt.xlabel("스텝")
plt.ylabel("플로 매칭 손실(MSE)")
plt.title("캐릭터 LoRA 학습 곡선")
plt.legend()
plt.show()

**결과 읽기**
- 50스텝 이동 평균이 처음 0.64 근처에서 0.55~0.70 사이를 오르내리며 끝까지 크게 줄지 않습니다. 손실은 그 스텝에 뽑힌 잡음 비율에 크게 좌우되므로, Diffusion 모델 LoRA는 손실로 학습 정도나 과적합을 판단하기 어렵습니다. 그래서 다음 소절과 6절처럼 그림으로 판단합니다.

### 5-5. 학습 중 그림 변화

같은 '우주 유영' 프롬프트와 같은 시드로, 체크포인트마다 그린 그림입니다.

In [ ]:
# 0스텝은 학습 전 그림입니다
snapshot_titles = ["학습 전"]
for step in checkpoint_steps:
    snapshot_titles.append(f"{step}스텝")
show_images(list(snapshots.values()), snapshot_titles, 5, "우주 유영 프롬프트, 시드 0")

**결과 읽기**
- **250스텝:** 학습 전의 파란 물고기가 크림색 얼굴과 분홍 볼을 가진 생물로 바뀌었지만, 접부채 귀와 초롱 꼬리는 아직 없습니다.
- **500·750스텝:** 접부채 귀, 초롱이 달린 꼬리, 왼쪽 볼의 점 세 개가 모두 나오고, 우주와 행성 배경도 그대로입니다.
- **1000스텝:** 초롱이는 남지만 몸이 작아지고 꼬리의 초롱이 줄었습니다. 장면보다 학습 그림의 모습을 따라가는 쪽으로 기울기 시작합니다.

## 6. 체크포인트 고르기

### 6-1. 체크포인트마다 확인 프롬프트 아홉

학습한 어댑터를 지우고, 저장한 체크포인트를 하나씩 파일에서 불러와 3절과 같은 프롬프트 아홉을 같은 시드 0으로 그립니다. 파일에서 불러와 그리므로, 저장한 어댑터 파일만으로 캐릭터가 나오는지도 함께 확인합니다(5-5의 중간 그림과 거의 같은 그림이 나옵니다. 학습 중에는 어댑터가 float32, 불러온 어댑터는 bf16이라 아주 작은 차이가 있습니다).

In [ ]:
# 학습에 쓴 어댑터("default")를 지웁니다. 이제 트랜스포머는 학습 전과 같습니다
transformer.delete_adapters("default")
transformer.eval()

checkpoint_images = {}
start_time = time.time()
for step in checkpoint_steps:
    adapter_name = f"step_{step:04d}"
    pipe.load_lora_weights(lora_root / adapter_name, adapter_name=adapter_name)  # 파일에서 얹고 이 어댑터만 켭니다
    images = {}
    for name, prompt in test_prompts.items():
        images[name] = generate(prompt, seed=0)
    checkpoint_images[step] = images
    pipe.delete_adapters(adapter_name)  # 다음 체크포인트를 얹기 전에 뺍니다
print(f"체크포인트 {len(checkpoint_steps)}개 x 그림 {len(test_prompts)}장: {(time.time() - start_time) / 60:.1f}분")

In [ ]:
rows = [("학습 전", before_images)]
for step in checkpoint_steps:
    rows.append((f"{step}스텝", checkpoint_images[step]))
compare_rows(rows, list(character_prompts), "캐릭터 확인 프롬프트, 같은 시드 0")
compare_rows(rows, list(general_prompts), "일반화 확인 프롬프트, 같은 시드 0")

**결과 읽기**
- **캐릭터:** 250스텝은 몸 색과 귀 모양만 조금 따라오고 초롱 꼬리가 없습니다. 500스텝부터 여섯 칸 모두에 초롱이의 특징이 나오고, 우주, 수박, 바닷속, 열기구 장면과 수채화, 연필 스케치 화풍도 프롬프트대로입니다. 1000스텝은 캐릭터가 작아지고 초롱이 줄어듭니다.
- **부류(작은 생물):** 250스텝부터 초롱이로 바뀝니다. 부류 낱말 `creature`가 캡션 42개에 모두 `Chorong the creature`로 들어 있어, 이 낱말의 뜻이 가장 먼저 초롱이 쪽으로 끌려갑니다.
- **다른 동물(개):** 250스텝까지는 평범한 개입니다. 500스텝부터 접부채 귀와 초롱 꼬리가 붙고, 1000스텝에는 개의 모습이 거의 사라지고 학습 그림의 숲·정원 배경까지 따라 들어옵니다. 스텝이 늘수록 일반화 능력이 무너지는 과적합입니다.
- **사진(사람):** 모든 체크포인트에서 카페의 실사 사진으로 남습니다(같은 시드여도 인물은 바뀝니다). 3D 애니메이션과 관계없는 프롬프트에는 어댑터의 영향이 작습니다.

### 6-2. 고르는 규칙과 결정

지난 시간 Tuned를 고른 규칙과 같은 생각입니다. 조건을 만족하는 체크포인트가 여럿이면 앞 체크포인트를 고릅니다. 더 학습한 체크포인트는 학습 그림을 더 외워 일반화 능력을 더 잃기 때문입니다.

1. **캐릭터:** 캐릭터 확인 여섯 칸에 캐릭터의 특징(크림색 둥근 몸, 접부채 모양 귀, 초롱이 달린 꼬리, 왼쪽 볼의 점 세 개)이 나오고, 장면과 화풍은 프롬프트대로 나와야 합니다. 이 조건을 만족하는 가장 앞 체크포인트를 고릅니다.
2. **일반화:** 일반화 확인 세 칸을 학습 전 그림과 견줘, 고른 체크포인트에서 무엇이 얼마나 새는지 확인합니다. 새는 만큼은 7절의 세기로 줄이고, 그래도 부족하면 9절의 prior preservation을 더합니다.

**결정: 500스텝.** 캐릭터 조건을 만족하는 가장 앞 체크포인트입니다(250스텝은 초롱 꼬리가 없습니다). 부류 낱말 `creature`는 이미 250스텝부터 초롱이로 바뀌어 체크포인트로는 피할 수 없고, 개에 초롱이 특징이 붙는 것은 500스텝부터 시작해 1000스텝으로 갈수록 심해지므로, 뒤 체크포인트보다 500스텝이 일반화 능력을 더 지킵니다.

## 7. 고른 어댑터 쓰기: 세기 조절

고른 체크포인트를 `chorong`이라는 이름으로 얹습니다. 파이프라인의 `set_adapters`는 어댑터 출력에 곱할 세기(weight)를 정합니다. 세기 0은 어댑터를 뺀 것과 같고, 0.5는 어댑터가 바꾼 양의 절반만 더합니다. 세기를 낮추면 캐릭터는 약해지는 대신 학습 전 모델의 그림에 가까워지므로, 일반화 능력을 지키는 또 하나의 손잡이입니다.

In [ ]:
chosen_step = 500  # 6-2의 규칙으로 고른 체크포인트
pipe.load_lora_weights(lora_root / f"step_{chosen_step:04d}", adapter_name="chorong")

strength_values = [0.0, 0.5, 0.8, 1.0]
strength_rows = []
for strength in strength_values:
    pipe.set_adapters(["chorong"], adapter_weights=[strength])
    images = {}
    for name in ["우주 유영", "수채화", "부류(작은 생물)", "다른 동물(개)"]:
        images[name] = generate(test_prompts[name], seed=0)
    strength_rows.append((f"세기 {strength}", images))
compare_rows(strength_rows, ["우주 유영", "수채화", "부류(작은 생물)", "다른 동물(개)"], f"{chosen_step}스텝 어댑터의 세기, 같은 시드 0")

**결과 읽기**
- **세기 0.5:** 우주 장면은 초록 생물, 수채화는 깃털 같은 귀의 생물이 되어 초롱이의 특징이 거의 사라집니다. 작은 생물은 크림색 토끼, 개는 평범한 개로 학습 전에 가깝습니다.
- **세기 0.8:** 우주와 수채화에 접부채 귀와 초롱 꼬리가 남습니다. 개는 개 모습으로 돌아오고 꼬리에 작은 초롱 하나만 남습니다. 작은 생물은 여전히 초롱이 모습입니다.
- **세기 1.0:** 개에도 접부채 귀가 붙습니다.
- 그래서 이 어댑터는 세기 0.8 정도로 씁니다. 캐릭터를 지키면서 개처럼 다른 대상에 새는 것을 줄입니다. 부류 낱말 `creature` 자체를 지키려면 세기로는 부족하고 9절의 방법이 필요합니다.

## 8. 내 프롬프트로 초롱이 그리기

고른 어댑터를 세기 0.8로 켜고, 원하는 장면을 적어 초롱이를 그립니다. 2-4에서 뺀 텍스트 인코더를 다시 얹으면 프롬프트 문장을 파이프라인에 바로 넣을 수 있습니다.

**프롬프트 쓰는 법**
- **트리거 단어:** `Chorong the creature`를 꼭 넣습니다. 귀, 꼬리, 볼의 점은 캡션에 적지 않고 트리거 단어에 묶였으므로 적지 않아도 나옵니다.
- **형식:** 학습 캡션과 같은 `a 3D animated cartoon of Chorong the creature` 뒤에 행동, 장소, 조명을 영어로 잇습니다. 화풍을 바꾸려면 앞부분을 `a watercolor painting of`처럼 바꿉니다(6절의 수채화, 연필 스케치).
- **순서:** FLUX.2는 앞에 나온 말에 더 주의를 줍니다. 꼭 나와야 할 것을 앞에 씁니다.
- **부정 프롬프트 없음:** FLUX.2는 빼 달라는 말을 따로 받지 않습니다. `no blur` 대신 `sharp focus`처럼 원하는 것을 적습니다.

In [ ]:
from transformers import Qwen3ForCausalLM

# 2-4에서 뺀 텍스트 인코더(약 8GB)를 다시 얹습니다. 이제 pipe 에 프롬프트 문장을 바로 넣습니다
pipe.text_encoder = Qwen3ForCausalLM.from_pretrained(model_name, subfolder="text_encoder", dtype=torch.bfloat16).to(device)

# 7절에서 정한 세기 0.8로 고른 어댑터를 켭니다
pipe.set_adapters(["chorong"], adapter_weights=[0.8])
print("GPU에 올라간 메모리(GB):", round(torch.cuda.memory_allocated() / 1024 ** 3, 1))

In [ ]:
# 그리고 싶은 장면을 영어로 적습니다(Chorong the creature 는 꼭 넣습니다)
my_prompt = "a 3D animated cartoon of Chorong the creature flying a red kite on a windy green hill at sunset"
n_images = 4  # 같은 프롬프트로 한 번에 만들 장 수(512 x 512 는 24GB GPU에서 2~4장)
seed = 0  # 바꾸면 다른 그림 묶음이 나옵니다

# num_images_per_prompt 만큼 시작 잡음을 따로 뽑아 한 번에 그립니다
generator = torch.Generator(device).manual_seed(seed)
start_time = time.time()
result = pipe(
    prompt=my_prompt,
    height=resolution,
    width=resolution,
    num_inference_steps=30,
    guidance_scale=4.0,
    num_images_per_prompt=n_images,
    generator=generator,
)
print(f"그림 {n_images}장: {time.time() - start_time:.0f}초")
show_images(result.images, [f"{i + 1}번" for i in range(n_images)], n_images, f"내 프롬프트, 시드 {seed}")

# 그림을 output 폴더에 저장합니다
for i, image in enumerate(result.images):
    image.save(output_dir / f"my_chorong_{i + 1}.png")

## 9. 필요할 때만: prior preservation

6절에서 본 것처럼 캡션, 체크포인트 고르기, 세기 조절로도 부류 낱말('creature')의 뜻은 캐릭터로 바뀌었습니다. 학습 뒤에도 트리거 단어 없는 'creature'를 평범하게 그려야 한다면 DreamBooth 논문의 prior preservation(사전 지식 보존)을 더합니다. 이 부록은 실습 시간 때문에 쓰지 않습니다(학습 시간이 두 배 가까이 늘고, 캐릭터를 늦게 배웁니다).

- **방법:** 학습 전 모델이 부류 프롬프트(`a 3D animated cartoon of a small creature`)로 그린 평범한 작은 생물 그림(부류 그림)을 수십~수백 장 미리 만들어 두고, 스텝마다 캐릭터 그림과 함께 학습합니다.
- **손실:** 캐릭터 그림 손실에 부류 그림 손실을 배율 $\lambda$(`prior_loss_weight`)만큼 더합니다.

$\displaystyle L = L_{\text{캐릭터}} + \lambda\, L_{\text{부류}}$

- **$\lambda$의 크기:** 부류 그림 손실은 '부류 프롬프트에는 학습 전 모델처럼 그려라'를 가르칩니다. $\lambda = 0$이면 이 부록의 학습과 같고, 클수록 일반화 능력을 더 지키는 대신 캐릭터를 늦게 배웁니다. 너무 크면 캐릭터를 거의 배우지 못합니다.
- **쓰는 법:** diffusers 공식 학습 스크립트(`train_dreambooth_lora_flux2_klein.py`)에 `--with_prior_preservation`, `--class_prompt`, `--num_class_images`, `--prior_loss_weight`를 주면 부류 그림 만들기와 두 손실 더하기를 스크립트가 합니다.